# FPL foundation-model and selection validation

This notebook runs the chronological 5k, 10k, and 20k context experiment, saves each completed model cell to Google Drive, evaluates matched likely-starter selections, and creates one downloadable results bundle.

Before running it, add `TABPFN_TOKEN` in Colab's **Secrets** panel and grant this notebook access. The token is read into memory and is never written to the results bundle. Select a GPU runtime for TabPFN and TabICL.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, sys

drive.mount('/content/drive')
PROJECT = Path('/content/fpl_performance_modeling')
DATA_REPO = Path('/content/Fantasy-Premier-League')
OUT = Path('/content/drive/MyDrive/fpl_colab_results')
OUT.mkdir(parents=True, exist_ok=True)

token = userdata.get('TABPFN_TOKEN')
if not token:
    raise RuntimeError('Add TABPFN_TOKEN in the Colab Secrets panel and grant notebook access.')
os.environ['TABPFN_TOKEN'] = token
os.environ['HF_HOME'] = str(OUT / 'model_cache' / 'huggingface')
os.environ['TABPFN_MODEL_CACHE_DIR'] = str(OUT / 'model_cache' / 'tabpfn')
os.environ['SKB_DATA_DIRECTORY'] = str(OUT / 'model_cache' / 'skrub')
print(sys.version)
print('Token loaded:', bool(os.environ.get('TABPFN_TOKEN')))

In [ ]:
import subprocess
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/2thleZ/fpl_performance_modeling.git', str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
if not DATA_REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', 'https://github.com/vaastav/Fantasy-Premier-League.git', str(DATA_REPO)], check=True)
subprocess.run(['git', '-C', str(DATA_REPO), 'sparse-checkout', 'set', 'data'], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project:', PROJECT)

## Run or resume the experiment

Re-running this cell skips successful checkpoints and retries failed cells. The default config runs direct models at 5k, 10k, and 20k; two-stage models at 5k; and full-history Ridge/CatBoost references.

In [ ]:
!python colab_experiment.py --data-root {DATA_REPO / 'data'} --out {OUT}

## Rebuild and download the bundle

Run this after stopping early or after resuming. It rebuilds all summaries from the completed checkpoints without fitting any models.

In [ ]:
!python colab_experiment.py --data-root {DATA_REPO / 'data'} --out {OUT} --bundle-only
from google.colab import files
files.download(str(OUT / 'fpl_colab_results.zip'))